In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse,Circle,FancyArrow
import matplotlib.colors as colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.font_manager import FontProperties
import parameters as p
import os
import re
import scipy
from importlib import reload
import pandas as pd
_ = reload(p)

In [ ]:
import os
import pandas as pd
import numpy as np

# =================setting=================
work_path = 'G:/percolation'
all_data_dir = os.path.join(work_path, "August2022")
data_dirs = os.listdir(all_data_dir)
data_dirs = [os.path.join(all_data_dir, d) for d in data_dirs]
pix_to_m = 0.003175
output_dir = os.path.join(work_path, "overlap_pair_data_6cols0")
os.makedirs(output_dir, exist_ok=True)


OVERLAP_RATIO_THRESHOLD = 0.7

xmin, xmax, ymin, ymax = 1.5774851, 18.9938216, 1.61291492, 9.5323406

def load_and_convert_csv(file_path):
    if not os.path.exists(file_path):
        return pd.DataFrame()
    df = pd.read_csv(file_path, header=None)
    if df.empty:
        return pd.DataFrame()
    
   
    df[0] = pd.to_numeric(df[0], errors='coerce') # Time
    df[1] = pd.to_numeric(df[1], errors='coerce') # X
    df[2] = pd.to_numeric(df[2], errors='coerce') # Y
    df = df.dropna(subset=[0, 1, 2])
    
    if df.empty:
        return pd.DataFrame()

    df[1] *= pix_to_m
    df[2] *= pix_to_m


    mask = (df[1] >= xmin) & (df[1] <= xmax) & (df[2] >= ymin) & (df[2] <= ymax)
    df = df[mask]
    
    df = df.sort_values(by=0).reset_index(drop=True)
    return df


for data_dir in data_dirs:
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('AB.csv') or f.endswith('BA.csv')]
    csv_files.sort()


    processed_count = 0
    skipped_count = 0

    for i, file1 in enumerate(csv_files):
        parts1 = file1.split('_')
        type1 = parts1[-1].split('.')[0]
        id1 = parts1[-2] 

        for file2 in csv_files[i+1:]:
            parts2 = file2.split('_')
            type2 = parts2[-1].split('.')[0]
            id2 = parts2[-2]

            if type1 != type2:
                continue

            df1 = load_and_convert_csv(os.path.join(data_dir, file1))
            df2 = load_and_convert_csv(os.path.join(data_dir, file2))

            if df1.empty or df2.empty:
                continue


            t1_min, t1_max = df1[0].min(), df1[0].max()
            t2_min, t2_max = df2[0].min(), df2[0].max()
            
            dur1 = t1_max - t1_min
            dur2 = t2_max - t2_min
            
            if dur1 == 0 or dur2 == 0:
                continue
        

            is_1_shorter = dur1 <= dur2
            if is_1_shorter:
                df_short, t_short_min, t_short_max, dur_short = df1, t1_min, t1_max, dur1
                df_long, t_long_min, t_long_max, dur_long = df2, t2_min, t2_max, dur2
                short_file, long_file = file1, file2
            else:
                df_short, t_short_min, t_short_max, dur_short = df2, t2_min, t2_max, dur2
                df_long, t_long_min, t_long_max, dur_long = df1, t1_min, t1_max, dur1
                short_file, long_file = file2, file1
            if dur_long > dur_short*3:
                continue
  
            overlap_start = max(t_short_min, t_long_min)
            overlap_end = min(t_short_max, t_long_max)
            
            if overlap_start >= overlap_end:
                continue
                
            overlap_dur = overlap_end - overlap_start
            
            ratio = overlap_dur / dur_short
            if ratio < OVERLAP_RATIO_THRESHOLD:
                skipped_count += 1
                continue


            start_short = t_short_min
            end_short = t_short_max

            target_dur = dur_short
           
            
            
            if overlap_start-t_short_min <= overlap_start-t_long_min:
                start_long = t_short_min
                end_long = t_short_min + target_dur
                mode = "Forward Enough"
            else:
                start_long = t_long_min
                end_long = t_long_min + target_dur
                mode = "Forward Insufficient -> Backward Fill"
                
           
            mask_short = (df_short[0] >= start_short) & (df_short[0] <= end_short)
            df_short_cut = df_short[mask_short].reset_index(drop=True)
            

            mask_long = (df_long[0] >= start_long) & (df_long[0] <= end_long)
            df_long_cut = df_long[mask_long].reset_index(drop=True)
            
            if df_short_cut.empty or df_long_cut.empty:
                continue

            len_short = len(df_short_cut)
            len_long = len(df_long_cut)
            min_len = min(len_short, len_long)
            
            if min_len == 0:
                continue
                
            df_short_final = df_short_cut.iloc[:min_len]
            df_long_final = df_long_cut.iloc[:min_len]
            
           
            if is_1_shorter:
                df1_out = df_short_final
                df2_out = df_long_final
            else:
                df1_out = df_long_final
                df2_out = df_short_final

            df1_out = df1_out.rename(columns={0: 'time1', 1: 'x1', 2: 'y1'})
            df2_out = df2_out.rename(columns={0: 'time2', 1: 'x2', 2: 'y2'})
            
            final_df = pd.concat([df1_out[['time1', 'x1', 'y1']], df2_out[['time2', 'x2', 'y2']]], axis=1)
            final_df = final_df[['time1', 'time2', 'x1', 'y1', 'x2', 'y2']]

            
            file_name = f"{id1}_{id2}_{type1}.csv"
            file_path = os.path.join(output_dir, file_name)
            
            final_df.to_csv(file_path, index=False)
            processed_count += 1
            

print(f"done")


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import random
import parameters as p
# ----------------- setting -----------------
work_path = 'G:/percolation'
data_dir = os.path.join(work_path, "overlap_pair_data_6cols0")  
output_dir = os.path.join(work_path, "data_analysis_results", "experiment_results")
os.makedirs(output_dir, exist_ok=True)
xmin, xmax, ymin, ymax = 1.5774851, 18.9938216, 1.61291492, 9.5323406
cell_size = p.target_thre*2
GRID_SIZE = (int(1/cell_size), int(1/cell_size))
# ----------------- calculation -----------------
def scale_x(x):
    return (x - xmin) / (xmax - xmin)
def scale_y(y):
    return (y - ymin) / (ymax - ymin)
def compute_overlap(df1, df2, r=cell_size):
    from sklearn.neighbors import NearestNeighbors
    pts1 = np.stack([scale_x(df1['x1']), scale_y(df1['y1'])], axis=1)
    pts2 = np.stack([scale_x(df2['x2']), scale_y(df2['y2'])], axis=1)

    
    nbrs12 = NearestNeighbors(n_neighbors=1).fit(pts2)
    dists12, _ = nbrs12.kneighbors(pts1)
    overlap1 = np.mean(dists12[:, 0] < r)

    
    nbrs21 = NearestNeighbors(n_neighbors=1).fit(pts1)
    dists21, _ = nbrs21.kneighbors(pts2)
    overlap2 = np.mean(dists21[:, 0] < r)

    
    return (overlap1 + overlap2) / 2.0

def compute_spreading(df):
    
    return np.sqrt((scale_x(df['x1']) - scale_x(df['x2']))**2 + (scale_y(df['y1']) - scale_y(df['y2']))**2)

def compute_coverage(df, grid_size=GRID_SIZE):
    
    xmin_, xmax_, ymin_, ymax_ = 0, 1, 0, 1
    H, _, _ = np.histogram2d(np.concatenate([scale_x(df['x1']), scale_x(df['x2'])]),
                              np.concatenate([scale_y(df['y1']), scale_y(df['y2'])]),
                              bins=grid_size, range=[[xmin_, xmax_], [ymin_, ymax_]])
    covered = np.sum(H > 0)
    total = grid_size[0] * grid_size[1]
    return covered / total

# ----------------- calcualte all files -----------------
overlaps, spreads, coverages = [], [], []


csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
random.seed(42)

for file in csv_files:
    file_path = os.path.join(data_dir, file)
    df = pd.read_csv(file_path)
    
    overlap = compute_overlap(df, df)  
    spread = compute_spreading(df)
    coverage = compute_coverage(df)
    
    overlaps.append(overlap)
    spreads.append(np.mean(spread))
    coverages.append(coverage)


fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(overlaps, spreads, coverages, color='b', alpha=0.7, edgecolor='k')

ax.set_xlabel("Overlap", fontsize=16)
ax.set_ylabel("Spreading", fontsize=16)
ax.set_zlabel("Coverage", fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
ax.tick_params(axis='z', labelsize=12)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "3D_overlap_spreading_coverage_experiment.svg"), dpi=500, bbox_inches="tight")
plt.close()


# Overlap vs Spreading
plt.figure(figsize=(8, 6))
plt.scatter(overlaps, spreads, color='b', alpha=0.7, edgecolor='k')
plt.xlabel("Overlap", fontsize=20)
plt.ylabel("Spreading", fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.legend(fontsize=18)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "overlap_vs_spreading_experiment.svg"), dpi=500, bbox_inches="tight")
plt.close()

# Overlap vs Coverage
plt.figure(figsize=(8, 6))
plt.scatter(overlaps, coverages, color='b', alpha=0.7, edgecolor='k')
plt.xlabel("Overlap", fontsize=20)
plt.ylabel("Coverage", fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.legend(fontsize=18)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "overlap_vs_coverage_experiment.svg"), dpi=500, bbox_inches="tight")
plt.close()

# Spreading vs Coverage
plt.figure(figsize=(8, 6))
plt.scatter(spreads, coverages, color='b', alpha=0.7, edgecolor='k')
plt.xlabel("Spreading", fontsize=20)
plt.ylabel("Coverage", fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.legend(fontsize=18)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "spreading_vs_coverage_experiment.svg"), dpi=500, bbox_inches="tight")
plt.close()


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering,KMeans
from sklearn.metrics import silhouette_score
from mpl_toolkits.mplot3d import Axes3D
all_data = np.vstack((overlaps, spreads, coverages)).T
# ----------------- 聚类分析 -----------------
def silhouette_method(data, max_clusters=10):
    silhouette_scores = []  # 保存每个聚类数的轮廓系数
    for n_clusters in range(2, max_clusters + 1):
    
        cluster = AgglomerativeClustering(n_clusters=n_clusters)
        cluster_labels = cluster.fit_predict(data)
        
        
        score = silhouette_score(data, cluster_labels)
        silhouette_scores.append(score)
    
   
    plt.figure(figsize=(8, 6))
    plt.plot(range(2, max_clusters + 1), silhouette_scores, marker='o', linestyle='--')
    plt.xlabel('Number of Clusters', fontsize=20)
    plt.ylabel('Silhouette Coefficient', fontsize=20)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "silhouette_score_clusters_experiment.svg"), dpi=500, bbox_inches="tight")
    plt.show()

    return silhouette_scores

silhouette_scores = silhouette_method(all_data, max_clusters=10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from kneed import KneeLocator  
import os

def elbow_method(data, max_clusters=10):
    sse = []
    cluster_nums = list(range(1, max_clusters + 1))

    for n_clusters in cluster_nums:
        if n_clusters == 1:
            
            center = np.mean(data, axis=0)
            sse_value = np.sum((data - center) ** 2)
        else:
            
            cluster = AgglomerativeClustering(n_clusters=n_clusters)
            labels = cluster.fit_predict(data)
            centers = np.array([data[labels == i].mean(axis=0) for i in range(n_clusters)])
            sse_value = np.sum([np.sum((data[labels == i] - centers[i]) ** 2) for i in range(n_clusters)])
        
        sse.append(sse_value)

   
    kneedle = KneeLocator(cluster_nums, sse, curve="convex", direction="decreasing")
    elbow = kneedle.elbow

    
    plt.figure(figsize=(8, 6))
    plt.plot(cluster_nums, sse, marker='o', linestyle='--')
    plt.xlabel('Number of Clusters', fontsize=20)
    plt.ylabel('SSE (Sum of Squared Errors)', fontsize=20)
    plt.xticks(np.arange(1,len(cluster_nums)+1), fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True)


    # plt.scatter(elbow, sse[elbow - 1], color='red', s=100, label=f'Elbow: {elbow} Clusters')
    # plt.legend(fontsize=12)
    plt.savefig(os.path.join(output_dir, "elbow_method_cluster_number_kneedle.svg"), dpi=500, bbox_inches="tight")

    return sse, elbow


all_data = np.array(all_data)


sse_values, elbow = elbow_method(all_data, max_clusters=10)

print(f"Best number of clusters (Elbow Method): {elbow}")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.spatial import ConvexHull, Delaunay
from sklearn.cluster import AgglomerativeClustering
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
work_path = 'G:/percolation'
output_dir = os.path.join(work_path, "data_analysis_results", "experiment_results")
os.makedirs(output_dir, exist_ok=True)

n_clusters = 3

# cluster order
orders = [1, 0, 2]
cluster_names = [
    'Territorial Division',
    'Direct Antagonism',
    'First-Exit Competition'
]
colors = ['green', 'purple', 'orange']


overlaps = np.array(overlaps)
spreads = np.array(spreads)
coverages = np.array(coverages)

X = np.vstack((overlaps, spreads, coverages)).T


agg = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
cluster_labels = agg.fit_predict(X)

# ================= Convex Hull Overlap =================
def compute_convex_hull_overlap(points1, points2, n_samples=100000, random_state=0):
    rng = np.random.default_rng(random_state)

    hull1 = ConvexHull(points1)
    hull2 = ConvexHull(points2)

    mins = np.minimum(points1.min(axis=0), points2.min(axis=0))
    maxs = np.maximum(points1.max(axis=0), points2.max(axis=0))

    samples = rng.uniform(mins, maxs, size=(n_samples, 3))

    d1 = Delaunay(points1[hull1.vertices])
    d2 = Delaunay(points2[hull2.vertices])

    in1 = d1.find_simplex(samples) >= 0
    in2 = d2.find_simplex(samples) >= 0

    union = np.sum(in1 | in2)
    inter = np.sum(in1 & in2)

    return 0.0 if union == 0 else inter / union

# ================= 3D Plot =================
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

points_per_cluster = []

for idx, cid in enumerate(orders):
    pts = X[cluster_labels == cid]
    points_per_cluster.append(pts)

    # scatter
    ax.scatter(
        pts[:, 0], pts[:, 1], pts[:, 2],
        color=colors[idx], label=cluster_names[idx],
        alpha=0.6, s=20
    )

    if len(pts) >= 4:
        hull = ConvexHull(pts)
        for simplex in hull.simplices:
            tri = Poly3DCollection([pts[simplex]])
            tri.set_color(colors[idx])
            tri.set_alpha(0.2)
            ax.add_collection3d(tri)

ax.set_xlabel("Overlap", fontsize=16)
ax.set_ylabel("Spreading", fontsize=16)
ax.set_zlabel("Coverage", fontsize=16)

ax.tick_params(axis='both', labelsize=12)
ax.tick_params(axis='z', labelsize=12)
from matplotlib.lines import Line2D
handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=colors[i], markersize=10,
           label=cluster_names[i], linestyle='')
    for i in range(len(cluster_names))
]

ax.legend(
    handles=handles,
    fontsize=16,
    bbox_to_anchor=(0.6, 1.02),
    loc="upper left"
)

plt.tight_layout()
plt.savefig(
    os.path.join(output_dir, "experiment_3D_convex_hull_wireframe.svg"),
    dpi=300, bbox_inches="tight"
)
plt.close()

print("3D convex hull (wireframe) saved.")

print("\nConvex hull overlap ratios (experimental data):")

for (i, pts1), (j, pts2) in combinations(enumerate(points_per_cluster), 2):
    overlap = compute_convex_hull_overlap(pts1, pts2)
    print(
        f"{cluster_names[i]} vs {cluster_names[j]}: "
        f"overlap ≈ {overlap:.4f}"
    )



In [ ]:
import os
from matplotlib import scale
import pandas as pd
import numpy as np
import io
import random
import base64
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

work_path = 'G:/percolation'
data_dir = os.path.join(work_path, "overlap_pair_data_6cols0")  
output_dir = os.path.join(work_path, "data_analysis_results", "experiment_results")
os.makedirs(output_dir, exist_ok=True)


xmin, xmax, ymin, ymax = 1.5774851, 18.9938216, 1.61291492, 9.5323406
cell_size = 0.1 
GRID_SIZE = (int(1/cell_size), int(1/cell_size))


def get_img_url(fig):
    buffer = io.BytesIO()
    plt.savefig(buffer, format='jpeg', bbox_inches='tight', facecolor='white')
    encoded_image = base64.b64encode(buffer.getvalue()).decode()
    im_url = "data:image/jpeg;base64," + encoded_image
    buffer.close()
    return im_url
def draw(x1,y1,x2,y2):
    fig = plt.figure(figsize=(4,4))
    ax = fig.add_subplot(111) 
    ax.scatter(x1,y1,c='b')
    ax.scatter(x2,y2,c='gold')
    #plt.gca().set_aspect("equal")
    plt.gca().set_xticklabels([])
    plt.gca().set_yticklabels([])
    plt.gca().tick_params(axis='both', which='both', length=0)
   
    plt.xlim([0,1])
    plt.ylim([0,1])

    img_url = get_img_url(fig)
    plt.close(fig)
    return img_url
def scale_x(x):
    return (x - xmin) / (xmax - xmin)
def scale_y(y):
    return (y - ymin) / (ymax - ymin)
def compute_overlap(df1, df2, r=cell_size):
   
    pts1 = np.stack([scale_x(df1['x1']), scale_y(df1['y1'])], axis=1)
    pts2 = np.stack([scale_x(df2['x2']), scale_y(df2['y2'])], axis=1)

    
    nbrs12 = NearestNeighbors(n_neighbors=1).fit(pts2)
    dists12, _ = nbrs12.kneighbors(pts1)
    overlap1 = np.mean(dists12[:, 0] < r)

   
    nbrs21 = NearestNeighbors(n_neighbors=1).fit(pts1)
    dists21, _ = nbrs21.kneighbors(pts2)
    overlap2 = np.mean(dists21[:, 0] < r)

    
    return (overlap1 + overlap2) / 2.0

def compute_spreading(df):
    
    return np.sqrt((scale_x(df['x1']) - scale_x(df['x2']))**2 + (scale_y(df['y1']) - scale_y(df['y2']))**2)

def compute_coverage(df, grid_size=GRID_SIZE):
    
    xmin_, xmax_, ymin_, ymax_ = 0, 1, 0, 1
    H, _, _ = np.histogram2d(np.concatenate([scale_x(df['x1']), scale_x(df['x2'])]),
                              np.concatenate([scale_y(df['y1']), scale_y(df['y2'])]),
                              bins=grid_size, range=[[xmin_, xmax_], [ymin_, ymax_]])
    covered = np.sum(H > 0)
    total = grid_size[0] * grid_size[1]
    return covered / total


overlaps, spreads, coverages, pair_ids, image_urls = [], [], [], [], []


csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
print(len(csv_files))
random.seed(42)

for i, file1 in enumerate(csv_files):
    
       
    id1 = file1.split('_')[0]
    id2 = file1.split('_')[1]

    
    df1 = pd.read_csv(os.path.join(data_dir, file1))
    
    
    overlap = compute_overlap(df1, df1)  
    spread = compute_spreading(df1)  
    coverage = compute_coverage(df1)  

   
    overlaps.append(overlap)
    spreads.append(np.mean(spread))
    coverages.append(coverage)
    pair_ids.append(f"{id1}-{id2}")



    img_url = draw(scale_x(df1['x1']), scale_y(df1['y1']), scale_x(df1['x2']), scale_y(df1['y2']))
   
    image_urls.append(img_url)


df_result = pd.DataFrame({
    'Pair ID': pair_ids,
    'Overlap': overlaps,  
    'Spreading': spreads,  
    'Coverage': coverages,  
    'Track Image URL': image_urls
})

output_excel_path = os.path.join(output_dir, 'comparison_results_overlap0.xlsx')
df_result.to_excel(output_excel_path, index=False)

print(f"Results saved to {output_excel_path}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
import random

# Define the function to calculate clustering quality scores
def clustering_quality_scores(X, labels):
    """Calculate internal quality metrics for unsupervised clustering."""
    scores = {}
    if len(set(labels)) > 1 and len(set(labels)) < len(X):
        scores['silhouette'] = silhouette_score(X, labels)
        scores['calinski_harabasz'] = calinski_harabasz_score(X, labels)
        scores['davies_bouldin'] = davies_bouldin_score(X, labels)
    else:
        scores['silhouette'] = np.nan
        scores['calinski_harabasz'] = np.nan
        scores['davies_bouldin'] = np.nan
    return scores

# Define function for clustering stability analysis
def clustering_stability(X, n_clusters=4, n_bootstrap=5, sample_frac=0.8, random_state=42):
    """
    Stability analysis: perform bootstrap sampling and calculate clustering consistency (ARI / NMI).
    """
    rng = np.random.default_rng(random_state)
    labels_list = []

    # Convert DataFrame to numpy array for clustering
    X = X.values  # If X is a pandas DataFrame, we convert it to a numpy array

    for b in range(n_bootstrap):
        # Bootstrap sampling
        idx = rng.choice(len(X), int(len(X) * sample_frac), replace=True)
        X_sample = X[idx]

        # Perform Agglomerative Clustering
        model = AgglomerativeClustering(n_clusters=n_clusters)
        labels = model.fit_predict(X_sample)
        labels_list.append((idx, labels))

    # Compare consistency pairwise
    ari_scores = []
    nmi_scores = []
    for i in range(len(labels_list)):
        for j in range(i + 1, len(labels_list)):
            idx_i, li = labels_list[i]
            idx_j, lj = labels_list[j]
            # Find common samples
            common_idx, idx_i_new, idx_j_new = np.intersect1d(idx_i, idx_j, return_indices=True)
            if len(common_idx) > 0:
                ari_scores.append(adjusted_rand_score(li[idx_i_new], lj[idx_j_new]))
                nmi_scores.append(normalized_mutual_info_score(li[idx_i_new], lj[idx_j_new]))

    return {
        "ARI_mean": np.mean(ari_scores) if ari_scores else np.nan,
        "NMI_mean": np.mean(nmi_scores) if nmi_scores else np.nan,
        "ARI_all": ari_scores,
        "NMI_all": nmi_scores,
    }


# ================== Load Experiment Data ==================
data_set_path = 'G:/percolation/data_analysis_results/experiment_results/comparison_results_overlap0.xlsx'
df = pd.read_excel(data_set_path)

# Step 1: Perform Agglomerative Clustering
X = df[['Overlap', 'Spreading', 'Coverage']]  # Using features for clustering
agg_clust = AgglomerativeClustering(n_clusters=3, linkage='ward')  # Set number of clusters
df['Cluster'] = agg_clust.fit_predict(X)

# Step 2: Define custom colors for each cluster
cluster_colors = {
    3: 'red',    # Cluster 0 - Red
    2: 'green',  # Cluster 1 - Green
    0: 'blue',   # Cluster 2 - Blue
    1: 'gold',   # Cluster 3 - Gold
}
# Map the cluster labels to colors
df['Cluster Color'] = df['Cluster'].map(cluster_colors)

# Step 3: Clustering Evaluation and Stability Analysis
# 1) Evaluate clustering quality
quality = clustering_quality_scores(X, df['Cluster'])
print("Clustering Quality (Silhouette, Calinski-Harabasz, Davies-Bouldin):")
for k, v in quality.items():
    print(f"  {k}: {v:.4f}")

# 2) Stability analysis (using bootstrap sampling)
stab = clustering_stability(X, n_clusters=3, n_bootstrap=100)
print("\nClustering Stability (Bootstrap 100 times):")
print(f"  Average ARI: {stab['ARI_mean']:.4f}")
print(f"  Average NMI: {stab['NMI_mean']:.4f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import LinearSegmentedColormap
#put on the grid (blue and yellow) remove the walls add some text target black
def floor_rects_metric_inline(csvfile, pix_to_m):
    T = pd.read_csv(csvfile)

    for idx, row in T.iterrows():
        x = row['x'] * pix_to_m
        y = row['y'] * pix_to_m
        w = row['width'] * pix_to_m
        h = row['height'] * pix_to_m

        plt.gca().add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=2, edgecolor='k', facecolor='none',linestyle='-'))
        plt.plot([xmin, xmin], [ymin,ymin+834.1352693*pix_to_m], color='white', linestyle='-', linewidth=2)
        plt.plot([xmax-971.4747*pix_to_m,xmax], [ymax,ymax], color='white', linestyle='-', linewidth=2)
        # if 'wall' in row['name']:
        #     plt.gca().add_patch(plt.Rectangle((x, y), w, h, linewidth=1, edgecolor=None, facecolor='gray'))
        if 'exh' in row['name']:
            plt.gca().add_patch(plt.Rectangle((x, y), w, h, linewidth=1, edgecolor=None, facecolor='black'))
def plot_ratio_blend_heatmap(grid_counts_blue, grid_counts_yellow, range, save_path, name, floor_csv, pix_to_m):
    fig, ax = plt.subplots(figsize=(8, 4))

    extent = range  # [xmin, xmax, ymin, ymax]
    xmin, xmax, ymin, ymax = extent

    total = grid_counts_blue + grid_counts_yellow
    ratio = np.zeros_like(total, dtype=float)
    mask = total > 0
    ratio[mask] = grid_counts_yellow[mask] / total[mask]

    cmap = LinearSegmentedColormap.from_list('BlueToYellow', ['blue', 'yellow'])
    colors_rgba = cmap(ratio)
    colors_rgba[~mask, 3] = 0  # transparent where no visits

    # Get grid dimensions
    ny, nx = ratio.shape  

    
    im = ax.imshow(colors_rgba, extent=extent, origin='lower', aspect='auto')

    
    x_grid_lines = np.linspace(xmin, xmax, nx + 1)
    y_grid_lines = np.linspace(ymin, ymax, ny + 1)

   

    ax.set_xticks([])
    ax.set_yticks([])

    # floor plan (walls, exits, etc.)
    floor_rects_metric_inline(floor_csv, pix_to_m)

    # Set axis limits
    ax.set_xlim(xmin-0.1, xmax+0.1)
    ax.set_ylim(ymin-0.1, ymax+0.1)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    

    plt.savefig(
        os.path.join(save_path, f"{name}_100pairs_trajectories.svg"),
        bbox_inches='tight'
    )
    plt.close()

work_path = 'G:/percolation'
pair_data_dir = os.path.join(work_path, "overlap_pair_data_6cols0")
data_set_path = 'G:/percolation/data_analysis_results/experiment_results/comparison_results_overlap0.xlsx'
df_p = pd.read_excel(data_set_path)  # Adjust path if needed

# Step 1: Perform Agglomerative Clustering
X = df_p[['Overlap', 'Spreading', 'Coverage']]  # Using features for clustering
agg_clust = AgglomerativeClustering(n_clusters=3, linkage='ward')  # Set number of clusters
df_p['Cluster'] = agg_clust.fit_predict(X)
output_dir = os.path.join(work_path, "data_analysis_results","experiment_results" ,"cluster_trajectory_occupancy")
os.makedirs(output_dir, exist_ok=True)

pix_to_m = 0.003175
xmin, xmax, ymin, ymax = 1.5774851, 18.9938216, 1.61291492, 9.5323406

N_PER_CLUSTER = 100
random.seed(42)

# ---------------- read cluster results----------------

clusters = sorted(df_p['Cluster'].unique())
floor_csv = os.path.join(
    work_path, "example_day_2022_10_02", "floor_overlay_labels.csv"
)

cluster_names = {
    2: 'Territorial Division',
    0: 'Direct Antagonism',
    1: 'First-Exit Competition'
}

for c in clusters:
    df_c = df_p[df_p['Cluster'] == c]

    sampled = df_c
    # .sample(
    #     min(N_PER_CLUSTER, len(df_c)),
    #     random_state=42
    # )

    
    
    nx, ny = 30, 15   
    x_edges = np.linspace(xmin, xmax, nx + 1)
    y_edges = np.linspace(ymin, ymax, ny + 1)

    grid_blue = np.zeros((nx, ny))
    grid_yellow = np.zeros((nx, ny))

    # trajectories
    for _, row in sampled.iterrows():
        pair_id = row['Pair ID'].replace('-', '_')
        csv_path = os.path.join(pair_data_dir, f"{pair_id}_AB.csv")

        if not os.path.exists(csv_path):
            continue

        df = pd.read_csv(csv_path)
        x1 = df['x1'].values
        y1 = df['y1'].values
        x2 = df['x2'].values
        y2 = df['y2'].values
        # ---------------- P1 (blue) ----------------
        

        ix1 = np.searchsorted(x_edges, x1, side='right') - 1
        iy1 = np.searchsorted(y_edges, y1, side='right') - 1

        valid1 = (ix1 >= 0) & (ix1 < nx) & (iy1 >= 0) & (iy1 < ny)
        for x, y in zip(ix1[valid1], iy1[valid1]):
            if np.mean(y1)>np.mean(y2):
                grid_blue[x, y] += 1
            else:
                grid_yellow[x, y] +=1

        # ---------------- P2 (yellow) ----------------
        

        ix2 = np.searchsorted(x_edges, x2, side='right') - 1
        iy2 = np.searchsorted(y_edges, y2, side='right') - 1

        valid2 = (ix2 >= 0) & (ix2 < nx) & (iy2 >= 0) & (iy2 < ny)
        for x, y in zip(ix2[valid2], iy2[valid2]):
            if np.mean(y1)>np.mean(y2):
                grid_yellow[x, y] += 1
            else:
                grid_blue[x, y] +=1

    plot_ratio_blend_heatmap(
            grid_counts_blue=grid_blue.T,
            grid_counts_yellow=grid_yellow.T,
            range=[xmin , xmax , ymin , ymax ],
            save_path=output_dir,
            name=cluster_names[c],
            floor_csv=floor_csv,
            pix_to_m=pix_to_m
        )

    

    print(f"Saved cluster {c} trajectory map.")

